# Engineering Intelligence Platform (EIP)

## 01. GitHub API Source Exploration

### 1. Objective

Explore the GitHub REST API to understand source schemas,
entity relationships, data availability and ingestion
requirements for engineering collaboration analytics.

### 2. Source Information

| Attribute | Description |
|---|---|
| Source System | GitHub |
| Source Type | REST API |
| Repository | apache/airflow |
| Authentication | Read-only personal access token |
| Response Format | JSON |
| Time Standard | UTC |

### 3. Source Inventory

| Source ID | Source Entity | Grain |
|---|---|---|
| SRC-001 | Repository Metadata | One record per repository |
| SRC-002 | Pull Requests | One record per pull request |
| SRC-003 | Pull Request Details | One record per pull request |
| SRC-004 | Pull Request Reviews | One record per review |

### 4. Exploration Scope

- API connectivity and authentication
- Source schema identification
- Primary and foreign key identification
- Nested JSON structures
- Source attribute availability
- API pagination and rate limits
- Preliminary data quality assessment
- Source reconciliation

### 5. Expected Outcomes

- Validated source inventory
- Source-to-target mapping requirements
- Bronze layer schema requirements
- Initial data quality findings
- Metric feasibility inputs

### 6. Security Considerations

- Credentials are loaded from a Git-ignored .env file.
- Authentication credentials are never printed or logged.
- Only public repository data is retrieved.
- Source payloads are treated as untrusted data.
- Raw source payloads are not printed in notebook outputs.

### 7. Reference Documentation

https://docs.github.com/en/rest

### 8. Development Phase

Phase 02: GitHub API Integration and Source Exploration


In [1]:
# ============================================================
# EIP | Notebook Environment Validation
# ============================================================

import sys

import httpx
import pandas as pd
from dotenv import dotenv_values

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("HTTPX version:", httpx.__version__)
print("Pandas version:", pd.__version__)
print("Environment validation: Successful")


Python executable: /Users/khalid/Projects/Engineering-Intelligence-Platform/.venv/bin/python
Python version: 3.14.10
HTTPX version: 0.28.1
Pandas version: 3.1.1
Environment validation: Successful


In [2]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 01: Imports and Environment Configuration
#
# Purpose:
# Initialize exploration dependencies, securely load source
# configuration and validate the execution environment.
# ============================================================

# ------------------------------------------------------------
# 01. Standard library imports
# ------------------------------------------------------------

import re
import sys
from pathlib import Path


# ------------------------------------------------------------
# 02. Third-party library imports
# ------------------------------------------------------------

import httpx
import pandas as pd
from dotenv import dotenv_values
from IPython.display import display


# ------------------------------------------------------------
# 03. Project root identification
# ------------------------------------------------------------

# Locate the project root without relying on a hardcoded
# machine-specific directory path.

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
        and (path / "backend").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Unable to locate the EIP project root.")


# ------------------------------------------------------------
# 04. Secure environment configuration
# ------------------------------------------------------------

ENV_FILE = PROJECT_ROOT / ".env"

if not ENV_FILE.is_file():
    raise FileNotFoundError("Local .env configuration is missing.")

config = dotenv_values(ENV_FILE)

GITHUB_TOKEN = config.get("GITHUB_TOKEN")
GITHUB_REPOSITORY = config.get("GITHUB_REPOSITORY")

if not GITHUB_TOKEN or not GITHUB_REPOSITORY:
    raise RuntimeError("Required GitHub configuration is missing.")

# Restrict the repository identifier to the owner/repository
# format before constructing API endpoint paths.

if not re.fullmatch(
    r"[A-Za-z0-9-]+/[A-Za-z0-9_.-]+",
    GITHUB_REPOSITORY,
):
    raise ValueError("Invalid GitHub repository identifier.")


# ------------------------------------------------------------
# 05. GitHub API configuration
# ------------------------------------------------------------

GITHUB_BASE_URL = "https://api.github.com"
GITHUB_API_VERSION = "2026-03-10"

REQUEST_TIMEOUT_SECONDS = 30.0

SAMPLE_PAGE_SIZE = 10
SAMPLE_PR_COUNT = 3
MAX_REVIEW_PAGES_PER_PR = 3

REQUEST_HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": GITHUB_API_VERSION,
}


# ------------------------------------------------------------
# 06. Pandas display configuration
# ------------------------------------------------------------

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)


# ------------------------------------------------------------
# 07. Environment validation
# ------------------------------------------------------------

# Never display the token, configuration dictionary or
# authentication request headers.

print("=" * 65)
print("EIP | GitHub Source Exploration")
print("=" * 65)

print("Project Root:", PROJECT_ROOT)
print("Python Executable:", sys.executable)
print("Python Version:", sys.version.split()[0])
print("HTTPX Version:", httpx.__version__)
print("Pandas Version:", pd.__version__)
print("Repository:", GITHUB_REPOSITORY)
print("GitHub API Version:", GITHUB_API_VERSION)
print("Authentication Configured:", bool(GITHUB_TOKEN))
print("Environment Validation: Successful")

print("=" * 65)

EIP | GitHub Source Exploration
Project Root: /Users/khalid/Projects/Engineering-Intelligence-Platform
Python Executable: /Users/khalid/Projects/Engineering-Intelligence-Platform/.venv/bin/python
Python Version: 3.14.10
HTTPX Version: 0.28.1
Pandas Version: 3.1.1
Repository: apache/airflow
GitHub API Version: 2026-03-10
Authentication Configured: True
Environment Validation: Successful


In [4]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 02: Reusable Exploration Functions
#
# Purpose:
# Standardize API communication, source schema inspection
# and preliminary data quality profiling.
#
# Note:
# These functions are intended for source exploration.
# Production retry and ingestion logic will be implemented
# separately within the data engineering layer.
# ============================================================


# ------------------------------------------------------------
# 01. GitHub API request function
# ------------------------------------------------------------

def fetch_github_data(
    endpoint: str,
    params: dict | None = None,
) -> httpx.Response:
    """
    Retrieve data from an approved GitHub repository endpoint.

    Parameters
    ----------
    endpoint:
        Relative GitHub REST API endpoint.

    params:
        Optional API query parameters.

    Returns
    -------
    httpx.Response
        Successful GitHub API response.
    """

    # Restrict requests to the configured repository.
    # Do not forward credentials to arbitrary URLs.

    allowed_endpoint = f"/repos/{GITHUB_REPOSITORY}"

    if not (
        endpoint == allowed_endpoint
        or endpoint.startswith(f"{allowed_endpoint}/")
    ):
        raise ValueError("Unsupported GitHub API endpoint.")

    with httpx.Client(
        base_url=GITHUB_BASE_URL,
        headers=REQUEST_HEADERS,
        timeout=REQUEST_TIMEOUT_SECONDS,
        follow_redirects=False,
    ) as client:

        response = client.get(
            endpoint,
            params=params,
        )

        # Stop execution if the source request fails.
        response.raise_for_status()

    return response


# ------------------------------------------------------------
# 02. API response metadata function
# ------------------------------------------------------------

def display_response_metadata(
    response: httpx.Response,
) -> None:
    """Display non-sensitive operational API metadata."""

    print("HTTP Status:", response.status_code)
    print(
        "API Rate Limit:",
        response.headers.get("x-ratelimit-limit"),
    )
    print(
        "Remaining Requests:",
        response.headers.get("x-ratelimit-remaining"),
    )
    print(
        "Additional Pages Available:",
        "next" in response.links,
    )


# ------------------------------------------------------------
# 03. Source schema profiling function
# ------------------------------------------------------------

def profile_source_schema(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Generate an attribute-level source profiling summary.

    Returns
    -------
    pd.DataFrame
        Source attribute names, inferred data types,
        non-null counts, null counts and null percentages.
    """

    record_count = len(dataframe)

    schema_profile = pd.DataFrame(
        {
            "sourceAttribute": dataframe.columns,
            "inferredDataType": dataframe.dtypes.astype(str).values,
            "nonNullCount": dataframe.notna().sum().values,
            "nullCount": dataframe.isna().sum().values,
        }
    )

    schema_profile["nullPct"] = (
        schema_profile["nullCount"] / record_count * 100
        if record_count > 0
        else None
    )

    schema_profile["nullPct"] = (
        schema_profile["nullPct"].round(2)
        if record_count > 0
        else None
    )

    return schema_profile


# ------------------------------------------------------------
# 04. Source attribute selection function
# ------------------------------------------------------------

def select_source_attributes(
    dataframe: pd.DataFrame,
    attributes: list[str],
) -> pd.DataFrame:
    """
    Select expected attributes while reporting missing columns.

    Missing attributes are retained as null for exploration.
    Production ingestion will enforce required fields.
    """

    missing_attributes = [
        attribute
        for attribute in attributes
        if attribute not in dataframe.columns
    ]

    print("Missing Expected Attributes:", missing_attributes)

    return dataframe.reindex(columns=attributes).copy()


print("Exploration functions initialized successfully.")

Exploration functions initialized successfully.


In [5]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 03: Repository Metadata Extraction
#
# Source ID: SRC-001
# Source Entity: GitHub Repositories
# Grain: One record per repository
# Source Endpoint: GET /repos/{owner}/{repo}
#
# Proposed Target:
# eip_tbl_bronze_githubRepositories_daily
#
# Purpose:
# Retrieve repository-level metadata and inspect available
# source attributes.
# ============================================================


# ------------------------------------------------------------
# 01. Source endpoint configuration
# ------------------------------------------------------------

repository_endpoint = f"/repos/{GITHUB_REPOSITORY}"


# ------------------------------------------------------------
# 02. Extract raw repository data
# ------------------------------------------------------------

repository_response = fetch_github_data(
    endpoint=repository_endpoint,
)

repository_raw = repository_response.json()

if not isinstance(repository_raw, dict):
    raise ValueError("Unexpected repository response structure.")


# ------------------------------------------------------------
# 03. Normalize nested JSON for exploration
# ------------------------------------------------------------

# Preserve the original payload separately from the flattened
# DataFrame used for source schema investigation.

repository_df = pd.json_normalize(
    repository_raw,
    sep=".",
)


# ------------------------------------------------------------
# 04. Select relevant business attributes
# ------------------------------------------------------------

repository_attributes = [
    "id",
    "name",
    "full_name",
    "owner.id",
    "owner.login",
    "description",
    "language",
    "default_branch",
    "created_at",
    "updated_at",
    "pushed_at",
    "stargazers_count",
    "forks_count",
    "open_issues_count",
    "archived",
    "private",
    "html_url",
]

repository_preview_df = select_source_attributes(
    repository_df,
    repository_attributes,
)


# ------------------------------------------------------------
# 05. Display extraction results
# ------------------------------------------------------------

print("\nEIP | SRC-001 Repository Metadata")
print("=" * 65)

display_response_metadata(repository_response)

print("\nRecords Retrieved:", len(repository_df))
print("Available Source Attributes:", len(repository_df.columns))

display(repository_preview_df.T)

Missing Expected Attributes: []

EIP | SRC-001 Repository Metadata
HTTP Status: 200
API Rate Limit: 5000
Remaining Requests: 4961
Additional Pages Available: False

Records Retrieved: 1
Available Source Attributes: 128


,0
id,33884891
name,airflow
full_name,apache/airflow
owner.id,47359
owner.login,apache
description,"Apache Airflow - A platform to programmatically author, schedule, and monitor workflows"
language,Python
default_branch,main
created_at,2015-04-13T18:04:58Z
updated_at,2026-11-26T10:00:49Z


In [6]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 04: Pull Request Extraction
#
# Source ID: SRC-002
# Source Entity: GitHub Pull Requests
# Grain: One record per pull request
#
# Proposed Target:
# eip_tbl_bronze_githubPullRequests_daily
#
# Purpose:
# Explore pull request activity, lifecycle timestamps,
# contributor information and pagination behavior.
# ============================================================


# ------------------------------------------------------------
# 01. Source endpoint configuration
# ------------------------------------------------------------

pull_requests_endpoint = f"/repos/{GITHUB_REPOSITORY}/pulls"

pull_requests_params = {
    "state": "all",
    "sort": "updated",
    "direction": "desc",
    "per_page": SAMPLE_PAGE_SIZE,
    "page": 1,
}


# ------------------------------------------------------------
# 02. Extract raw pull request records
# ------------------------------------------------------------

# Retrieve both open and closed pull requests to investigate
# the availability of different lifecycle timestamps.

pull_requests_response = fetch_github_data(
    endpoint=pull_requests_endpoint,
    params=pull_requests_params,
)

pull_requests_raw = pull_requests_response.json()

if not isinstance(pull_requests_raw, list):
    raise ValueError("Unexpected pull request response structure.")

if not pull_requests_raw:
    raise RuntimeError("No pull requests returned for exploration.")


# ------------------------------------------------------------
# 03. Normalize nested JSON for exploration
# ------------------------------------------------------------

# Preserve the original source payload separately for
# future Bronze layer processing.

pull_requests_df = pd.json_normalize(
    pull_requests_raw,
    sep=".",
)


# ------------------------------------------------------------
# 04. Select relevant source attributes
# ------------------------------------------------------------

pull_request_attributes = [
    "id",
    "number",
    "state",
    "draft",
    "user.id",
    "user.login",
    "user.type",
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
    "author_association",
    "base.ref",
    "head.ref",
    "html_url",
]

pull_requests_preview_df = select_source_attributes(
    pull_requests_df,
    pull_request_attributes,
)


# ------------------------------------------------------------
# 05. Display extraction results
# ------------------------------------------------------------

print("\nEIP | SRC-002 Pull Requests")
print("=" * 65)

display_response_metadata(pull_requests_response)

print("\nRecords Retrieved:", len(pull_requests_df))
print("Available Source Attributes:", len(pull_requests_df.columns))

display(pull_requests_preview_df)


# ------------------------------------------------------------
# 06. Inspect pull request state distribution
# ------------------------------------------------------------

print("\nPull Request State Distribution:")

pull_request_state_summary_df = (
    pull_requests_preview_df["state"]
    .value_counts(dropna=False)
    .rename_axis("pullRequestState")
    .reset_index(name="recordCount")
)

display(pull_request_state_summary_df)


# ------------------------------------------------------------
# 07. Inspect merge status
# ------------------------------------------------------------

print("\nMerged Pull Requests:")

print(
    pull_requests_preview_df["merged_at"].notna().sum()
)

print("\nUnmerged Pull Requests:")

print(
    pull_requests_preview_df["merged_at"].isna().sum()
)

Missing Expected Attributes: []

EIP | SRC-002 Pull Requests
HTTP Status: 200
API Rate Limit: 5000
Remaining Requests: 4960
Additional Pages Available: True

Records Retrieved: 10
Available Source Attributes: 361


,id,number,state,draft,user.id,user.login,user.type,created_at,updated_at,closed_at,merged_at,author_association,base.ref,head.ref,html_url
0,5494312066,77598,open,False,49793253,ahnazary,User,2026-11-25T15:29:40Z,2026-11-26T10:46:28Z,NaN,NaN,CONTRIBUTOR,main,fix-rtif-cleanup-attempt-history,https://github.com/apache/airflow/pull/77598
1,5272088932,76967,open,True,68415893,jason810496,User,2026-11-12T05:28:41Z,2026-11-26T10:44:53Z,NaN,NaN,MEMBER,main,feature/lang-sdk/taskflow-stub-mapped-sdk-binders,https://github.com/apache/airflow/pull/76967
2,4148360837,70570,open,True,68415893,jason810496,User,2026-07-28T04:53:37Z,2026-11-26T10:44:50Z,NaN,NaN,MEMBER,main,feature/lang-sdk/taskflow-stub-dag-mapped,https://github.com/apache/airflow/pull/70570
3,4049905904,69849,open,False,99777196,viiccwen,User,2026-07-14T05:29:04Z,2026-11-26T10:42:19Z,NaN,NaN,CONTRIBUTOR,main,optimize-asset-event-association-memory,https://github.com/apache/airflow/pull/69849
4,5184288121,76575,open,False,35884252,amoghrajesh,User,2026-11-06T07:09:14Z,2026-11-26T10:40:21Z,NaN,NaN,CONTRIBUTOR,main,registry-additions-for-durable-exec,https://github.com/apache/airflow/pull/76575
5,5431473529,77455,open,False,246779673,Ayushrawatcy009,User,2026-11-21T19:13:08Z,2026-11-26T10:39:25Z,NaN,NaN,NONE,main,alt-click-extra-links,https://github.com/apache/airflow/pull/77455
6,5504449797,77620,open,False,85313829,sjyangkevin,User,2026-11-26T05:16:43Z,2026-11-26T10:33:58Z,NaN,NaN,CONTRIBUTOR,main,fab-plugins-app-session-backend,https://github.com/apache/airflow/pull/77620
7,5505694791,77624,closed,False,75422783,estella00911,User,2026-11-26T07:15:43Z,2026-11-26T10:31:18Z,2026-11-26T09:36:32Z,2026-11-26T09:36:32Z,CONTRIBUTOR,main,zh-TW-translation-gaps,https://github.com/apache/airflow/pull/77624
8,5294585240,77015,open,False,35884252,amoghrajesh,User,2026-11-13T11:22:26Z,2026-11-26T10:30:54Z,NaN,NaN,CONTRIBUTOR,main,resume-on-retry,https://github.com/apache/airflow/pull/77015
9,5508263159,77632,open,False,4018068,yakovlef,User,2026-11-26T10:25:54Z,2026-11-26T10:25:59Z,NaN,NaN,CONTRIBUTOR,main,docs-kafka-shared-stream,https://github.com/apache/airflow/pull/77632



Pull Request State Distribution:


,pullRequestState,recordCount
0,open,9
1,closed,1



Merged Pull Requests:
1

Unmerged Pull Requests:
9


In [7]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 05: Pull Request Details Extraction
#
# Source ID: SRC-003
# Source System: GitHub REST API
# Source Entity: Pull Request Details
# Source Endpoint: GET /repos/{owner}/{repo}/pulls/{number}
# Grain: One detailed record per pull request per extraction
#
# Proposed Bronze Target:
# eip_tbl_bronze_githubPullRequestDetails_daily
#
# Proposed Silver Target:
# eip_tbl_silver_pullRequests_daily
#
# Purpose:
# Enrich pull request records with code change statistics,
# commit counts, collaboration attributes and merge details.
#
# Dependencies:
# SRC-001: Repository Metadata
# SRC-002: Pull Requests
#
# Exploration Strategy:
# Select different PR lifecycle conditions to investigate
# attribute availability and source data quality.
# ============================================================


# ------------------------------------------------------------
# 01. Prepare pull request selection dataset
# ------------------------------------------------------------

# Work on a copy to preserve the original source dataset.

pr_selection_df = pull_requests_preview_df.copy()

# Convert timestamps to UTC for chronological comparisons.

pr_selection_df["created_at"] = pd.to_datetime(
    pr_selection_df["created_at"],
    utc=True,
    errors="raise",
)

pr_selection_df["merged_at"] = pd.to_datetime(
    pr_selection_df["merged_at"],
    utc=True,
    errors="raise",
)


# ------------------------------------------------------------
# 02. Identify different pull request lifecycle conditions
# ------------------------------------------------------------

# Select the most recently merged pull request.

merged_prs_df = (
    pr_selection_df.loc[
        pr_selection_df["merged_at"].notna()
    ]
    .sort_values("merged_at", ascending=False)
)


# Select the oldest open, non-draft pull request.
# This allows exploration of long-running PRs.

open_prs_df = (
    pr_selection_df.loc[
        (pr_selection_df["state"] == "open")
        & (pr_selection_df["draft"] == False)
    ]
    .sort_values("created_at", ascending=True)
)


# Select the oldest open draft pull request.
# Draft PRs may follow a different review lifecycle.

draft_prs_df = (
    pr_selection_df.loc[
        (pr_selection_df["state"] == "open")
        & (pr_selection_df["draft"] == True)
    ]
    .sort_values("created_at", ascending=True)
)


# ------------------------------------------------------------
# 03. Consolidate the exploration sample
# ------------------------------------------------------------

selected_prs_df = pd.concat(
    [
        merged_prs_df.head(1).assign(
            selectionReason="Recently Merged PR"
        ),
        open_prs_df.head(1).assign(
            selectionReason="Oldest Sampled Open PR"
        ),
        draft_prs_df.head(1).assign(
            selectionReason="Oldest Sampled Draft PR"
        ),
    ],
    ignore_index=True,
)

# Fall back to other available PRs when a lifecycle
# category is not present in the source sample.

selected_prs_df = (
    pd.concat(
        [
            selected_prs_df,
            pr_selection_df.assign(
                selectionReason="Additional Sample PR"
            ),
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset=["number"])
    .head(SAMPLE_PR_COUNT)
    .reset_index(drop=True)
)

sample_pr_numbers = (
    selected_prs_df["number"]
    .astype(int)
    .tolist()
)

if not sample_pr_numbers:
    raise RuntimeError("No valid pull requests selected.")


# ------------------------------------------------------------
# 04. Display selected pull requests
# ------------------------------------------------------------

print("EIP | Pull Request Sample Selection")
print("=" * 65)

display(
    selected_prs_df[
        [
            "id",
            "number",
            "state",
            "draft",
            "created_at",
            "merged_at",
            "selectionReason",
        ]
    ]
)

print("Selected Pull Requests:", sample_pr_numbers)


# ------------------------------------------------------------
# 05. Build source identifier lookup
# ------------------------------------------------------------

# GitHub PR numbers are repository-specific, while source
# PR IDs provide stable identifiers for downstream joins.

pr_source_id_lookup = {
    record["number"]: record["id"]
    for record in pull_requests_raw
}


# ------------------------------------------------------------
# 06. Extract detailed pull request payloads
# ------------------------------------------------------------

pull_request_details_raw = []
detail_extraction_results = []

for pr_number in sample_pr_numbers:

    detail_endpoint = (
        f"/repos/{GITHUB_REPOSITORY}/pulls/{pr_number}"
    )

    detail_response = fetch_github_data(
        endpoint=detail_endpoint,
    )

    detail_payload = detail_response.json()

    # Validate the expected source response structure.

    if not isinstance(detail_payload, dict):
        raise ValueError(
            f"Unexpected response structure for PR #{pr_number}."
        )

    # Reconcile the returned PR number with the request.

    if detail_payload.get("number") != pr_number:
        raise ValueError(
            f"Pull request number mismatch for PR #{pr_number}."
        )

    # Reconcile the stable GitHub identifier with SRC-002.

    if detail_payload.get("id") != pr_source_id_lookup[pr_number]:
        raise ValueError(
            f"Source identifier mismatch for PR #{pr_number}."
        )

    # Preserve the complete original source payload.
    # No transformation is applied to the raw record.

    pull_request_details_raw.append(detail_payload)

    # Capture non-sensitive extraction metadata.

    detail_extraction_results.append(
        {
            "pullRequestNumber": pr_number,
            "pullRequestId": detail_payload["id"],
            "httpStatus": detail_response.status_code,
            "identifierMatch": True,
            "remainingApiRequests": (
                detail_response.headers.get(
                    "x-ratelimit-remaining"
                )
            ),
            "extractionStatus": "SUCCESS",
        }
    )

    print(
        f"PR #{pr_number}: "
        "Extraction and reconciliation successful"
    )


# ------------------------------------------------------------
# 07. Normalize detailed source payloads
# ------------------------------------------------------------

# Flatten nested JSON for exploration only.
# The original payload remains available for future
# Bronze layer persistence.

pull_request_details_df = pd.json_normalize(
    pull_request_details_raw,
    sep=".",
)


# ------------------------------------------------------------
# 08. Select relevant source attributes
# ------------------------------------------------------------

pull_request_detail_attributes = [

    # Source identifiers
    "id",
    "number",

    # Current lifecycle status
    "state",
    "draft",
    "merged",

    # Lifecycle timestamps
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",

    # Code change statistics
    "additions",
    "deletions",
    "changed_files",
    "commits",

    # Collaboration statistics
    "comments",
    "review_comments",

    # Merge account information
    "merged_by.id",
    "merged_by.login",
]

pull_request_details_preview_df = select_source_attributes(
    pull_request_details_df,
    pull_request_detail_attributes,
)


# ------------------------------------------------------------
# 09. Display source extraction results
# ------------------------------------------------------------

print("\nEIP | SRC-003 Pull Request Details")
print("=" * 65)

display_response_metadata(detail_response)

print(
    "\nRecords Retrieved:",
    len(pull_request_details_df),
)

print(
    "Available Source Attributes:",
    len(pull_request_details_df.columns),
)

print(
    "Selected Source Attributes:",
    len(pull_request_detail_attributes),
)

display(pull_request_details_preview_df)


# ------------------------------------------------------------
# 10. Display code change and collaboration statistics
# ------------------------------------------------------------

print("\nCode Change and Collaboration Statistics:")

display(
    pull_request_details_preview_df[
        [
            "number",
            "state",
            "draft",
            "additions",
            "deletions",
            "changed_files",
            "commits",
            "comments",
            "review_comments",
        ]
    ]
)


# ------------------------------------------------------------
# 11. Display source schema profile
# ------------------------------------------------------------

print("\nSelected Source Attribute Profile:")

pull_request_details_schema_df = profile_source_schema(
    pull_request_details_preview_df,
)

display(pull_request_details_schema_df)


# ------------------------------------------------------------
# 12. Display extraction and reconciliation summary
# ------------------------------------------------------------

detail_extraction_summary_df = pd.DataFrame(
    detail_extraction_results
)

print("\nPull Request Details Extraction Summary:")

display(detail_extraction_summary_df)

print(
    "All Requested Records Retrieved:",
    len(pull_request_details_raw) == len(sample_pr_numbers),
)

print(
    "All Source Identifiers Reconciled:",
    bool(
        detail_extraction_summary_df["identifierMatch"].all()
    ),
)

EIP | Pull Request Sample Selection


,id,number,state,draft,created_at,merged_at,selectionReason
0,5505694791,77624,closed,False,2026-11-26 07:15:43+00:00,2026-11-26 09:36:32+00:00,Recently Merged PR
1,4049905904,69849,open,False,2026-07-14 05:29:04+00:00,NaT,Oldest Sampled Open PR
2,4148360837,70570,open,True,2026-07-28 04:53:37+00:00,NaT,Oldest Sampled Draft PR


Selected Pull Requests: [77624, 69849, 70570]
PR #77624: Extraction and reconciliation successful
PR #69849: Extraction and reconciliation successful
PR #70570: Extraction and reconciliation successful
Missing Expected Attributes: []

EIP | SRC-003 Pull Request Details
HTTP Status: 200
API Rate Limit: 5000
Remaining Requests: 4957
Additional Pages Available: False

Records Retrieved: 3
Available Source Attributes: 392
Selected Source Attributes: 17


,id,number,state,draft,merged,created_at,updated_at,closed_at,merged_at,additions,deletions,changed_files,commits,comments,review_comments,merged_by.id,merged_by.login
0,5505694791,77624,closed,False,True,2026-11-26T07:15:43Z,2026-11-26T10:31:18Z,2026-11-26T09:36:32Z,2026-11-26T09:36:32Z,88,1,7,2,0,3,108824346.0,henry3260
1,4049905904,69849,open,False,False,2026-07-14T05:29:04Z,2026-11-26T10:42:19Z,NaN,NaN,397,45,2,7,5,20,NaN,NaN
2,4148360837,70570,open,True,False,2026-07-28T04:53:37Z,2026-11-26T10:44:50Z,NaN,NaN,5243,411,48,39,5,14,NaN,NaN



Code Change and Collaboration Statistics:


,number,state,draft,additions,deletions,changed_files,commits,comments,review_comments
0,77624,closed,False,88,1,7,2,0,3
1,69849,open,False,397,45,2,7,5,20
2,70570,open,True,5243,411,48,39,5,14



Selected Source Attribute Profile:


,sourceAttribute,inferredDataType,nonNullCount,nullCount,nullPct
0,id,int64,3,0,0.00
1,number,int64,3,0,0.00
2,state,str,3,0,0.00
3,draft,bool,3,0,0.00
4,merged,bool,3,0,0.00
5,created_at,str,3,0,0.00
6,updated_at,str,3,0,0.00
7,closed_at,str,1,2,66.67
8,merged_at,str,1,2,66.67
9,additions,int64,3,0,0.00



Pull Request Details Extraction Summary:


,pullRequestNumber,pullRequestId,httpStatus,identifierMatch,remainingApiRequests,extractionStatus
0,77624,5505694791,200,True,4959,SUCCESS
1,69849,4049905904,200,True,4958,SUCCESS
2,70570,4148360837,200,True,4957,SUCCESS


All Requested Records Retrieved: True
All Source Identifiers Reconciled: True


In [8]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 06: Pull Request Reviews Extraction
#
# Source ID: SRC-004
# Source System: GitHub REST API
# Source Entity: Pull Request Reviews
# Source Endpoint:
# GET /repos/{owner}/{repo}/pulls/{number}/reviews
#
# Grain: One review record per pull request per extraction
#
# Proposed Bronze Target:
# eip_tbl_bronze_githubReviews_daily
#
# Proposed Silver Target:
# eip_tbl_silver_reviews_daily
#
# Purpose:
# Retrieve review activity, reviewer information, review
# states and submission timestamps to support engineering
# collaboration and pull request lifecycle analytics.
#
# Dependencies:
# SRC-001: Repository Metadata
# SRC-002: Pull Requests
# SRC-003: Pull Request Details
# ============================================================


# ------------------------------------------------------------
# 01. Initialize extraction containers
# ------------------------------------------------------------

reviews_raw = []
review_context_records = []
review_extraction_results = []

# Associate repository-specific PR numbers with stable
# GitHub pull request identifiers.

pr_id_lookup = {
    record["number"]: record["id"]
    for record in pull_request_details_raw
}

repository_id = repository_raw["id"]

print("Repository:", GITHUB_REPOSITORY)
print("Selected Pull Requests:", sample_pr_numbers)


# ------------------------------------------------------------
# 02. Extract review records
# ------------------------------------------------------------

for pr_number in sample_pr_numbers:

    reviews_endpoint = (
        f"/repos/{GITHUB_REPOSITORY}"
        f"/pulls/{pr_number}/reviews"
    )

    review_count = 0
    pages_retrieved = 0
    has_next_page = True

    # Bound the number of API requests during exploration.
    # Production ingestion will retrieve all required pages.

    for page_number in range(
        1,
        MAX_REVIEW_PAGES_PER_PR + 1,
    ):

        reviews_response = fetch_github_data(
            endpoint=reviews_endpoint,
            params={
                "per_page": 100,
                "page": page_number,
            },
        )

        review_payload = reviews_response.json()

        # Validate the expected source response structure.

        if not isinstance(review_payload, list):
            raise ValueError(
                f"Unexpected review response for PR #{pr_number}."
            )

        for review in review_payload:

            if not isinstance(review, dict):
                raise ValueError("Invalid review record.")

            # Preserve the original source payload.

            reviews_raw.append(review)

            # Attach parent entity context separately.
            # The raw review payload remains unchanged.

            review_context_records.append(
                {
                    "repositoryId": repository_id,
                    "pullRequestId": pr_id_lookup[pr_number],
                    "pullRequestNumber": pr_number,
                    "review": review,
                }
            )

        review_count += len(review_payload)
        pages_retrieved += 1

        # Inspect API pagination metadata to determine
        # whether additional review records are available.

        has_next_page = "next" in reviews_response.links

        if not has_next_page:
            break

    # Record extraction completeness independently of
    # the number of reviews returned.
    #
    # A PR with zero reviews is not equivalent to a PR
    # whose reviews were only partially extracted.

    review_extraction_results.append(
        {
            "pullRequestNumber": pr_number,
            "pullRequestId": pr_id_lookup[pr_number],
            "reviewsRetrieved": review_count,
            "pagesRetrieved": pages_retrieved,
            "isExtractionComplete": not has_next_page,
            "remainingApiRequests": (
                reviews_response.headers.get(
                    "x-ratelimit-remaining"
                )
            ),
        }
    )

    print(
        f"PR #{pr_number}: "
        f"{review_count} reviews retrieved "
        f"across {pages_retrieved} page(s)"
    )


# ------------------------------------------------------------
# 03. Normalize nested review records
# ------------------------------------------------------------

reviews_df = pd.json_normalize(
    review_context_records,
    sep=".",
)


# ------------------------------------------------------------
# 04. Define selected analytical attributes
# ------------------------------------------------------------

review_attributes = [

    # Parent entity identifiers
    "repositoryId",
    "pullRequestId",
    "pullRequestNumber",

    # Review identifier
    "review.id",

    # Reviewer information
    "review.user.id",
    "review.user.login",
    "review.user.type",

    # Review lifecycle information
    "review.state",
    "review.submitted_at",

    # Collaboration and commit information
    "review.author_association",
    "review.commit_id",

    # Source evidence reference
    "review.html_url",
]

reviews_preview_df = select_source_attributes(
    reviews_df,
    review_attributes,
)


# ------------------------------------------------------------
# 05. Preserve identifier precision
# ------------------------------------------------------------

# Construct nullable integer columns directly from the
# original payloads to avoid float conversion when source
# identifiers contain null values.

reviews_preview_df["review.id"] = pd.array(
    [
        record["review"].get("id")
        for record in review_context_records
    ],
    dtype="Int64",
)

reviews_preview_df["review.user.id"] = pd.array(
    [
        (record["review"].get("user") or {}).get("id")
        for record in review_context_records
    ],
    dtype="Int64",
)


# ------------------------------------------------------------
# 06. Display source extraction results
# ------------------------------------------------------------

print("\nEIP | SRC-004 Pull Request Reviews")
print("=" * 65)

print("Records Retrieved:", len(reviews_df))

print(
    "Available Source and Context Attributes:",
    len(reviews_df.columns),
)

print(
    "Selected Source and Context Attributes:",
    len(review_attributes),
)

print(
    "Remaining API Requests:",
    reviews_response.headers.get("x-ratelimit-remaining"),
)

display(reviews_preview_df)


# ------------------------------------------------------------
# 07. Inspect review state distribution
# ------------------------------------------------------------

# Review states represent source-level review records.
# They must not be interpreted directly as current PR
# approval status without additional business logic.

review_state_summary_df = (
    reviews_preview_df["review.state"]
    .value_counts(dropna=False)
    .rename_axis("reviewState")
    .reset_index(name="recordCount")
)

print("\nReview State Distribution:")

display(review_state_summary_df)


# ------------------------------------------------------------
# 08. Generate review activity by pull request
# ------------------------------------------------------------

# Count submitted reviews separately from reviews that
# do not contain a submission timestamp.

review_activity_df = (
    reviews_preview_df
    .assign(
        isSubmitted=(
            reviews_preview_df["review.submitted_at"].notna()
        )
    )
    .groupby("pullRequestNumber", as_index=False)
    .agg(
        reviewRecords=("review.id", "size"),
        submittedReviews=("isSubmitted", "sum"),
        uniqueReviewerAccounts=("review.user.id", "nunique"),
    )
)

# Include all selected PRs, including those without reviews.

review_extraction_summary_df = pd.DataFrame(
    review_extraction_results,
)

review_activity_summary_df = (
    review_extraction_summary_df
    .merge(
        review_activity_df,
        on="pullRequestNumber",
        how="left",
    )
)

activity_columns = [
    "reviewRecords",
    "submittedReviews",
    "uniqueReviewerAccounts",
]

review_activity_summary_df[activity_columns] = (
    review_activity_summary_df[activity_columns]
    .fillna(0)
    .astype("Int64")
)

print("\nReview Activity by Pull Request:")

display(review_activity_summary_df)


# ------------------------------------------------------------
# 09. Reconcile review activity with PR details
# ------------------------------------------------------------

# General comments, review comments and submitted reviews
# represent different forms of collaboration activity.

pr_collaboration_df = (
    pull_request_details_preview_df[
        [
            "number",
            "state",
            "draft",
            "comments",
            "review_comments",
        ]
    ]
    .rename(
        columns={
            "number": "pullRequestNumber",
            "comments": "generalComments",
            "review_comments": "codeReviewComments",
        }
    )
    .merge(
        review_activity_summary_df[
            [
                "pullRequestNumber",
                "reviewRecords",
                "submittedReviews",
                "uniqueReviewerAccounts",
                "isExtractionComplete",
            ]
        ],
        on="pullRequestNumber",
        how="left",
        validate="one_to_one",
    )
)

print("\nConsolidated Pull Request Collaboration:")

display(pr_collaboration_df)


# ------------------------------------------------------------
# 10. Profile selected review attributes
# ------------------------------------------------------------

reviews_schema_profile_df = profile_source_schema(
    reviews_preview_df,
)

print("\nSelected Review Attribute Profile:")

display(reviews_schema_profile_df)


# ------------------------------------------------------------
# 11. Validate review extraction completeness
# ------------------------------------------------------------

duplicate_review_ids = int(
    reviews_preview_df["review.id"]
    .dropna()
    .duplicated()
    .sum()
)

null_review_ids = int(
    reviews_preview_df["review.id"]
    .isna()
    .sum()
)

all_reviews_complete = bool(
    review_extraction_summary_df[
        "isExtractionComplete"
    ].all()
)

print("\nEIP | Review Extraction Validation")
print("=" * 65)

print("Pull Requests Processed:", len(sample_pr_numbers))
print("Review Records Retrieved:", len(reviews_raw))
print("Null Review Identifiers:", null_review_ids)
print("Duplicate Review Identifiers:", duplicate_review_ids)
print("All Review Extractions Complete:", all_reviews_complete)

print(
    "Raw and Context Records Reconciled:",
    len(reviews_raw) == len(review_context_records),
)

Repository: apache/airflow
Selected Pull Requests: [77624, 69849, 70570]
PR #77624: 4 reviews retrieved across 1 page(s)
PR #69849: 16 reviews retrieved across 1 page(s)
PR #70570: 2 reviews retrieved across 1 page(s)
Missing Expected Attributes: []

EIP | SRC-004 Pull Request Reviews
Records Retrieved: 22
Available Source and Context Attributes: 35
Selected Source and Context Attributes: 12
Remaining API Requests: 4954


,repositoryId,pullRequestId,pullRequestNumber,review.id,review.user.id,review.user.login,review.user.type,review.state,review.submitted_at,review.author_association,review.commit_id,review.html_url
0,33884891,5505694791,77624,6070031750,5144808,Lee-W,User,APPROVED,2026-11-26T07:48:53Z,MEMBER,2485e9fd3ce88e3e5ef48307dd7d42211d82a85d,https://github.com/apache/airflow/pull/77624#pullrequestreview-6070031750
1,33884891,5505694791,77624,6070704697,75422783,estella00911,User,COMMENTED,2026-11-26T08:29:11Z,CONTRIBUTOR,2485e9fd3ce88e3e5ef48307dd7d42211d82a85d,https://github.com/apache/airflow/pull/77624#pullrequestreview-6070704697
2,33884891,5505694791,77624,6070741984,108824346,henry3260,User,APPROVED,2026-11-26T08:31:43Z,MEMBER,33d83d314b01c898e0cea439baaa44a880c564b6,https://github.com/apache/airflow/pull/77624#pullrequestreview-6070741984
3,33884891,5505694791,77624,6070958810,4927361,FrankYang0529,User,APPROVED,2026-11-26T08:49:00Z,MEMBER,33d83d314b01c898e0cea439baaa44a880c564b6,https://github.com/apache/airflow/pull/77624#pullrequestreview-6070958810
4,33884891,4049905904,69849,4693033539,5144808,Lee-W,User,COMMENTED,2026-07-14T09:55:55Z,MEMBER,NaN,https://github.com/apache/airflow/pull/69849#pullrequestreview-4693033539
5,33884891,4049905904,69849,4695258591,99777196,viiccwen,User,COMMENTED,2026-07-14T14:20:57Z,CONTRIBUTOR,NaN,https://github.com/apache/airflow/pull/69849#pullrequestreview-4695258591
6,33884891,4049905904,69849,4695557093,99777196,viiccwen,User,COMMENTED,2026-07-14T14:50:34Z,CONTRIBUTOR,NaN,https://github.com/apache/airflow/pull/69849#pullrequestreview-4695557093
7,33884891,4049905904,69849,4712836534,5144808,Lee-W,User,DISMISSED,2026-07-16T10:38:26Z,MEMBER,NaN,https://github.com/apache/airflow/pull/69849#pullrequestreview-4712836534
8,33884891,4049905904,69849,4892036628,605277,uranusjr,User,COMMENTED,2026-08-09T17:56:01Z,MEMBER,NaN,https://github.com/apache/airflow/pull/69849#pullrequestreview-4892036628
9,33884891,4049905904,69849,4892040316,605277,uranusjr,User,COMMENTED,2026-08-09T17:58:02Z,MEMBER,NaN,https://github.com/apache/airflow/pull/69849#pullrequestreview-4892040316



Review State Distribution:


,reviewState,recordCount
0,COMMENTED,18
1,APPROVED,3
2,DISMISSED,1



Review Activity by Pull Request:


,pullRequestNumber,pullRequestId,reviewsRetrieved,pagesRetrieved,isExtractionComplete,remainingApiRequests,reviewRecords,submittedReviews,uniqueReviewerAccounts
0,77624,5505694791,4,1,True,4956,4,4,4
1,69849,4049905904,16,1,True,4955,16,16,3
2,70570,4148360837,2,1,True,4954,2,2,2



Consolidated Pull Request Collaboration:


,pullRequestNumber,state,draft,generalComments,codeReviewComments,reviewRecords,submittedReviews,uniqueReviewerAccounts,isExtractionComplete
0,77624,closed,False,0,3,4,4,4,True
1,69849,open,False,5,20,16,16,3,True
2,70570,open,True,5,14,2,2,2,True



Selected Review Attribute Profile:


,sourceAttribute,inferredDataType,nonNullCount,nullCount,nullPct
0,repositoryId,int64,22,0,0.00
1,pullRequestId,int64,22,0,0.00
2,pullRequestNumber,int64,22,0,0.00
3,review.id,Int64,22,0,0.00
4,review.user.id,Int64,22,0,0.00
5,review.user.login,str,22,0,0.00
6,review.user.type,str,22,0,0.00
7,review.state,str,22,0,0.00
8,review.submitted_at,str,22,0,0.00
9,review.author_association,str,22,0,0.00



EIP | Review Extraction Validation
Pull Requests Processed: 3
Review Records Retrieved: 22
Null Review Identifiers: 0
Duplicate Review Identifiers: 0
All Review Extractions Complete: True
Raw and Context Records Reconciled: True


In [9]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 07: Consolidated Source Data Profiling
#
# Purpose:
# Consolidate source schemas, attribute availability,
# inferred data types and preliminary data quality results
# across all four GitHub source datasets.
#
# Dependencies:
# SRC-001: Repository Metadata
# SRC-002: Pull Requests
# SRC-003: Pull Request Details
# SRC-004: Pull Request Reviews
#
# Outputs:
# - Source-level extraction inventory
# - Attribute-level schema profile
# - Missing attribute report
# - Null value profile
# - Preliminary nullability assessment
#
# Scope:
# Sample-level profiling only. Results must not be
# interpreted as full historical population statistics.
# ============================================================


# ------------------------------------------------------------
# 01. Preserve nullable source identifier precision
# ------------------------------------------------------------

# Pandas inferred merged_by.id as float64 because the
# sampled records contain both integer IDs and null values.
#
# Reconstruct the column directly from the raw JSON
# rather than converting potentially rounded floats.

pull_request_details_preview_df["merged_by.id"] = pd.array(
    [
        (record.get("merged_by") or {}).get("id")
        for record in pull_request_details_raw
    ],
    dtype="Int64",
)


# ------------------------------------------------------------
# 02. Register source datasets
# ------------------------------------------------------------

# The original DataFrames contain the discovered flattened
# source attributes. Preview DataFrames contain the selected
# attributes intended for downstream processing.

source_datasets = {
    "SRC-001": {
        "sourceEntity": "Repositories",
        "original": repository_df,
        "selected": repository_preview_df,
    },
    "SRC-002": {
        "sourceEntity": "Pull Requests",
        "original": pull_requests_df,
        "selected": pull_requests_preview_df,
    },
    "SRC-003": {
        "sourceEntity": "Pull Request Details",
        "original": pull_request_details_df,
        "selected": pull_request_details_preview_df,
    },
    "SRC-004": {
        "sourceEntity": "Pull Request Reviews",
        "original": reviews_df,
        "selected": reviews_preview_df,
    },
}


# ------------------------------------------------------------
# 03. Define preliminary required source attributes
# ------------------------------------------------------------

# Required attributes are essential for identifying records
# or establishing valid analytical relationships.
#
# These rules are preliminary and will later be documented
# and enforced through production schema validation.

required_source_attributes = {
    "SRC-001": {
        "id",
        "name",
        "full_name",
        "owner.id",
        "owner.login",
        "created_at",
    },
    "SRC-002": {
        "id",
        "number",
        "state",
        "created_at",
        "updated_at",
    },
    "SRC-003": {
        "id",
        "number",
        "state",
        "merged",
        "created_at",
        "updated_at",
    },
    "SRC-004": {
        "repositoryId",
        "pullRequestId",
        "pullRequestNumber",
        "review.id",
        "review.state",
    },
}


# ------------------------------------------------------------
# 04. Document known nullable attributes
# ------------------------------------------------------------

# Nullability must be evaluated using source semantics,
# rather than treating every missing value as an error.

nullable_attribute_reasons = {
    "SRC-002": {
        "closed_at": "May be null for an open PR.",
        "merged_at": "May be null for an unmerged PR.",
        "user.id": "Author account may be unavailable.",
        "user.login": "Author account may be unavailable.",
        "user.type": "Author account may be unavailable.",
    },
    "SRC-003": {
        "closed_at": "May be null for an open PR.",
        "merged_at": "May be null for an unmerged PR.",
        "merged_by.id": "May be null for an unmerged PR.",
        "merged_by.login": "May be null for an unmerged PR.",
    },
    "SRC-004": {
        "review.submitted_at": "May be null for pending reviews.",
        "review.commit_id": "Associated commit may be unavailable.",
        "review.user.id": "Reviewer account may be unavailable.",
        "review.user.login": "Reviewer account may be unavailable.",
        "review.user.type": "Reviewer account may be unavailable.",
    },
}


# ------------------------------------------------------------
# 05. Initialize profiling result containers
# ------------------------------------------------------------

source_profiles = []
source_inventory_records = []


# ------------------------------------------------------------
# 06. Profile each source dataset
# ------------------------------------------------------------

for source_id, source_config in source_datasets.items():

    source_entity = source_config["sourceEntity"]

    original_df = source_config["original"]
    selected_df = source_config["selected"]

    source_profile_df = profile_source_schema(selected_df)

    # Attach source identifiers for consolidated reporting.

    source_profile_df.insert(
        0,
        "sourceId",
        source_id,
    )

    source_profile_df.insert(
        1,
        "sourceEntity",
        source_entity,
    )

    # Identify attributes observed in the flattened source
    # schema, rather than columns introduced by reindexing.
    # Presence means observed in at least one sample record.

    source_profile_df["isAttributePresent"] = (
        source_profile_df["sourceAttribute"]
        .isin(original_df.columns)
    )

    # Identify attributes required for analytical processing.

    source_profile_df["isRequired"] = (
        source_profile_df["sourceAttribute"]
        .isin(required_source_attributes[source_id])
    )

    # Attach documented reasons for potentially valid nulls.

    source_profile_df["nullableReason"] = (
        source_profile_df["sourceAttribute"]
        .map(nullable_attribute_reasons.get(source_id, {}))
    )

    source_profiles.append(source_profile_df)

    # Capture source-level record and attribute counts.

    source_inventory_records.append(
        {
            "sourceId": source_id,
            "sourceEntity": source_entity,
            "recordCount": len(original_df),
            "discoveredAttributes": len(original_df.columns),
            "selectedAttributes": len(selected_df.columns),
            "availableSelectedAttributes": int(
                source_profile_df["isAttributePresent"].sum()
            ),
        }
    )


# ------------------------------------------------------------
# 07. Consolidate attribute-level profiling results
# ------------------------------------------------------------

source_schema_profile_df = pd.concat(
    source_profiles,
    ignore_index=True,
)


# ------------------------------------------------------------
# 08. Classify preliminary attribute quality
# ------------------------------------------------------------

def classify_attribute_quality(row: pd.Series) -> str:
    """
    Classify sample-level source attribute quality.

    Status definitions:
        REQUIRED_ATTRIBUTE_MISSING:
            Required attribute absent from the source schema.

        ATTRIBUTE_NOT_OBSERVED:
            Non-required attribute absent from the sample.

        REQUIRED_VALUE_MISSING:
            Required attribute contains one or more nulls.

        EXPECTED_NULL_POSSIBLE:
            Attribute contains nulls with a known valid reason.

        REVIEW_REQUIRED:
            Attribute contains nulls without a documented reason.

        NOT_ASSESSED:
            No sample records are available.

        PASS:
            Attribute is present and contains no observed nulls.
    """

    if row["nonNullCount"] + row["nullCount"] == 0:
        return "NOT_ASSESSED"

    if not row["isAttributePresent"]:

        if row["isRequired"]:
            return "REQUIRED_ATTRIBUTE_MISSING"

        return "ATTRIBUTE_NOT_OBSERVED"

    if row["nullCount"] == 0:
        return "PASS"

    if row["isRequired"]:
        return "REQUIRED_VALUE_MISSING"

    if pd.notna(row["nullableReason"]):
        return "EXPECTED_NULL_POSSIBLE"

    return "REVIEW_REQUIRED"


source_schema_profile_df["qualityStatus"] = (
    source_schema_profile_df.apply(
        classify_attribute_quality,
        axis=1,
    )
)


# ------------------------------------------------------------
# 09. Generate source-level extraction inventory
# ------------------------------------------------------------

source_inventory_df = pd.DataFrame(
    source_inventory_records,
)

source_inventory_df["missingSelectedAttributes"] = (
    source_inventory_df["selectedAttributes"]
    - source_inventory_df["availableSelectedAttributes"]
)

print("EIP | Consolidated Source Inventory")
print("=" * 65)

display(source_inventory_df)


# ------------------------------------------------------------
# 10. Display consolidated source attribute profile
# ------------------------------------------------------------

print("\nEIP | Consolidated Source Schema Profile")
print("=" * 65)

display(source_schema_profile_df)


# ------------------------------------------------------------
# 11. Identify missing source attributes
# ------------------------------------------------------------

source_missing_attributes_df = (
    source_schema_profile_df.loc[
        ~source_schema_profile_df["isAttributePresent"]
    ]
    .reset_index(drop=True)
)

print("\nMissing Expected Source Attributes:")

display(source_missing_attributes_df)


# ------------------------------------------------------------
# 12. Identify attributes containing null values
# ------------------------------------------------------------

source_null_profile_df = (
    source_schema_profile_df.loc[
        (source_schema_profile_df["nullCount"] > 0)
        & source_schema_profile_df["isAttributePresent"]
    ]
    .sort_values(
        ["sourceId", "nullPct"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print("\nSource Attributes Containing Null Values:")

display(
    source_null_profile_df[
        [
            "sourceId",
            "sourceAttribute",
            "inferredDataType",
            "nullCount",
            "nullPct",
            "nullableReason",
            "qualityStatus",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Generate preliminary data quality summary
# ------------------------------------------------------------

source_quality_summary_df = (
    source_schema_profile_df
    .groupby(
        ["sourceId", "qualityStatus"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "attributeCount"})
)

print("\nEIP | Preliminary Data Quality Summary")
print("=" * 65)

display(source_quality_summary_df)


# ------------------------------------------------------------
# 14. Identify critical schema and completeness failures
# ------------------------------------------------------------

critical_quality_statuses = [
    "REQUIRED_ATTRIBUTE_MISSING",
    "REQUIRED_VALUE_MISSING",
]

critical_quality_issues_df = (
    source_schema_profile_df.loc[
        source_schema_profile_df["qualityStatus"]
        .isin(critical_quality_statuses)
    ]
    .reset_index(drop=True)
)

print("\nCritical Attribute Quality Issues:")

display(critical_quality_issues_df)


# ------------------------------------------------------------
# 15. Display overall profiling results
# ------------------------------------------------------------

print("\nEIP | Source Profiling Results")
print("=" * 65)

print(
    "Source Datasets Profiled:",
    len(source_datasets),
)

print(
    "Total Source Records:",
    int(source_inventory_df["recordCount"].sum()),
)

print(
    "Total Discovered Attributes:",
    int(
        source_inventory_df["discoveredAttributes"].sum()
    ),
)

print(
    "Total Selected Attributes:",
    len(source_schema_profile_df),
)

print(
    "Missing Selected Attributes:",
    len(source_missing_attributes_df),
)

print(
    "Attributes Containing Null Values:",
    len(source_null_profile_df),
)

print(
    "Critical Attribute Quality Issues:",
    len(critical_quality_issues_df),
)

print(
    "Required Attribute Checks Passed:",
    critical_quality_issues_df.empty
    and all(
        not item["selected"].empty
        for item in source_datasets.values()
    ),
)

EIP | Consolidated Source Inventory


,sourceId,sourceEntity,recordCount,discoveredAttributes,selectedAttributes,availableSelectedAttributes,missingSelectedAttributes
0,SRC-001,Repositories,1,128,17,17,0
1,SRC-002,Pull Requests,10,361,15,15,0
2,SRC-003,Pull Request Details,3,392,17,17,0
3,SRC-004,Pull Request Reviews,22,35,12,12,0



EIP | Consolidated Source Schema Profile


,sourceId,sourceEntity,sourceAttribute,inferredDataType,nonNullCount,nullCount,nullPct,isAttributePresent,isRequired,nullableReason,qualityStatus
0,SRC-001,Repositories,id,int64,1,0,0.00,True,True,NaN,PASS
1,SRC-001,Repositories,name,str,1,0,0.00,True,True,NaN,PASS
2,SRC-001,Repositories,full_name,str,1,0,0.00,True,True,NaN,PASS
3,SRC-001,Repositories,owner.id,int64,1,0,0.00,True,True,NaN,PASS
4,SRC-001,Repositories,owner.login,str,1,0,0.00,True,True,NaN,PASS
5,SRC-001,Repositories,description,str,1,0,0.00,True,False,NaN,PASS
6,SRC-001,Repositories,language,str,1,0,0.00,True,False,NaN,PASS
7,SRC-001,Repositories,default_branch,str,1,0,0.00,True,False,NaN,PASS
8,SRC-001,Repositories,created_at,str,1,0,0.00,True,True,NaN,PASS
9,SRC-001,Repositories,updated_at,str,1,0,0.00,True,False,NaN,PASS



Missing Expected Source Attributes:


,sourceId,sourceEntity,sourceAttribute,inferredDataType,nonNullCount,nullCount,nullPct,isAttributePresent,isRequired,nullableReason,qualityStatus


... (3 outputs left out)

,sourceId,qualityStatus,attributeCount
0,SRC-001,PASS,17
1,SRC-002,EXPECTED_NULL_POSSIBLE,2
2,SRC-002,PASS,13
3,SRC-003,EXPECTED_NULL_POSSIBLE,4
4,SRC-003,PASS,13
5,SRC-004,EXPECTED_NULL_POSSIBLE,1
6,SRC-004,PASS,11



Critical Attribute Quality Issues:


,sourceId,sourceEntity,sourceAttribute,inferredDataType,nonNullCount,nullCount,nullPct,isAttributePresent,isRequired,nullableReason,qualityStatus



EIP | Source Profiling Results
Source Datasets Profiled: 4
Total Source Records: 36
Total Discovered Attributes: 916
Total Selected Attributes: 61
Missing Selected Attributes: 0
Attributes Containing Null Values: 7
Critical Attribute Quality Issues: 0
Required Attribute Checks Passed: True


In [11]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 08: Source Reconciliation and Metric Feasibility
#
# Purpose:
# Validate source identifiers, entity relationships,
# lifecycle consistency and review extraction completeness.
#
# Dependencies:
# SRC-001: Repository Metadata
# SRC-002: Pull Requests
# SRC-003: Pull Request Details
# SRC-004: Pull Request Reviews
#
# Outputs:
# - Source reconciliation results
# - Conditional data quality results
# - Pull request lifecycle analytical prototype
# - Metric feasibility assessment
#
# Proposed Gold Target:
# eip_tbl_gold_prLifecycle_wide
#
# Note:
# All calculations are exploratory and performed in memory.
# ============================================================


# ------------------------------------------------------------
# 01. Initialize data quality validation framework
# ------------------------------------------------------------

validation_results = []


def record_validation(
    rule_id: str,
    rule_name: str,
    records_checked: int,
    failed_records: int,
    severity: str = "ERROR",
) -> None:
    """
    Record the result of a source reconciliation rule.

    A rule with no eligible records is not considered passed.
    Warning-level failures require investigation but do not
    necessarily indicate corrupted source data.
    """

    if records_checked == 0:
        status = "NOT_ASSESSED"
    elif failed_records == 0:
        status = "PASS"
    elif severity == "WARNING":
        status = "WARNING"
    else:
        status = "FAIL"

    validation_results.append(
        {
            "ruleId": rule_id,
            "ruleName": rule_name,
            "severity": severity,
            "recordsChecked": records_checked,
            "failedRecords": failed_records,
            "validationStatus": status,
        }
    )


# ------------------------------------------------------------
# 02. Validate primary source identifiers
# ------------------------------------------------------------

identifier_config = [
    (
        "DQ-001",
        "Repository identifier",
        repository_preview_df,
        "id",
    ),
    (
        "DQ-002",
        "Pull request identifier",
        pull_requests_preview_df,
        "id",
    ),
    (
        "DQ-003",
        "Pull request detail identifier",
        pull_request_details_preview_df,
        "id",
    ),
    (
        "DQ-004",
        "Review identifier",
        reviews_preview_df,
        "review.id",
    ),
]

for rule_id, rule_name, dataframe, identifier in identifier_config:

    # Detect records with missing or duplicate identifiers.

    invalid_identifiers = (
        dataframe[identifier].isna()
        | dataframe[identifier].duplicated(keep=False)
    )

    record_validation(
        rule_id=rule_id,
        rule_name=rule_name,
        records_checked=len(dataframe),
        failed_records=int(invalid_identifiers.sum()),
    )


# ------------------------------------------------------------
# 03. Reconcile source relationships
# ------------------------------------------------------------

# Every detailed PR must exist in the sampled PR list.

invalid_detail_relationships = (
    ~pull_request_details_preview_df["id"]
    .isin(pull_requests_preview_df["id"])
)

record_validation(
    rule_id="DQ-005",
    rule_name="PR details to pull requests",
    records_checked=len(pull_request_details_preview_df),
    failed_records=int(invalid_detail_relationships.sum()),
)


# Every review must reference the expected parent PR.

review_parent_lookup = (
    pull_request_details_preview_df
    .set_index("number")["id"]
)

expected_parent_ids = (
    reviews_preview_df["pullRequestNumber"]
    .map(review_parent_lookup)
)

invalid_review_relationships = (
    reviews_preview_df["pullRequestId"].isna()
    | expected_parent_ids.isna()
    | reviews_preview_df["pullRequestId"]
        .ne(expected_parent_ids)
        .fillna(True)
)

record_validation(
    rule_id="DQ-006",
    rule_name="Reviews to parent pull requests",
    records_checked=len(reviews_preview_df),
    failed_records=int(invalid_review_relationships.sum()),
)


# Validate repository context attached during review ingestion.

invalid_repository_relationships = (
    reviews_preview_df["repositoryId"].isna()
    | reviews_preview_df["repositoryId"].ne(
        repository_raw["id"]
    )
)

record_validation(
    rule_id="DQ-007",
    rule_name="Reviews to parent repository",
    records_checked=len(reviews_preview_df),
    failed_records=int(invalid_repository_relationships.sum()),
)


# ------------------------------------------------------------
# 04. Reconcile overlapping PR source attributes
# ------------------------------------------------------------

# List and detail endpoints contain overlapping attributes.
# Compare their values to identify potential source drift.

overlapping_attributes = [
    "state",
    "draft",
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
]

pr_reconciliation_df = (
    pull_requests_preview_df[
        ["id", *overlapping_attributes]
    ]
    .merge(
        pull_request_details_preview_df[
            ["id", *overlapping_attributes]
        ],
        on="id",
        how="inner",
        suffixes=("_list", "_detail"),
        validate="one_to_one",
    )
)

attribute_reconciliation_results = []

for attribute in overlapping_attributes:

    list_values = pr_reconciliation_df[
        f"{attribute}_list"
    ]

    detail_values = pr_reconciliation_df[
        f"{attribute}_detail"
    ]

    # Two null values represent a matching observation.

    matching_values = (
        list_values.eq(detail_values).fillna(False)
        | (
            list_values.isna()
            & detail_values.isna()
        )
    )

    attribute_reconciliation_results.append(
        {
            "sourceAttribute": attribute,
            "recordsCompared": len(pr_reconciliation_df),
            "matchingRecords": int(matching_values.sum()),
            "mismatchedRecords": int(
                (~matching_values).sum()
            ),
        }
    )

pr_attribute_reconciliation_df = pd.DataFrame(
    attribute_reconciliation_results
)

# Differences can occur because the endpoints are queried
# at different times. Treat them as potential source drift.

record_validation(
    rule_id="DQ-008",
    rule_name="PR list and detail attribute consistency",
    records_checked=int(
        pr_attribute_reconciliation_df[
            "recordsCompared"
        ].sum()
    ),
    failed_records=int(
        pr_attribute_reconciliation_df[
            "mismatchedRecords"
        ].sum()
    ),
    severity="WARNING",
)

print("EIP | PR Attribute Reconciliation")
print("=" * 65)

display(pr_attribute_reconciliation_df)


# ------------------------------------------------------------
# 05. Prepare standardized lifecycle datasets
# ------------------------------------------------------------

# Use copies to preserve the original source DataFrames.

pr_lifecycle_df = (
    pull_request_details_preview_df
    .copy()
    .rename(
        columns={
            "id": "pullRequestId",
            "number": "pullRequestNumber",
            "state": "pullRequestState",
            "draft": "isDraft",
            "merged": "isMerged",
            "created_at": "createdAt",
            "updated_at": "updatedAt",
            "closed_at": "closedAt",
            "merged_at": "mergedAt",
            "additions": "additionsCount",
            "deletions": "deletionsCount",
            "changed_files": "changedFilesCount",
            "commits": "commitsCount",
        }
    )
)

# Obtain author information from the original PR list.

pr_author_df = (
    pull_requests_preview_df[
        ["id", "user.id", "user.type"]
    ]
    .rename(
        columns={
            "id": "pullRequestId",
            "user.id": "authorId",
            "user.type": "authorType",
        }
    )
)

pr_lifecycle_df = pr_lifecycle_df.merge(
    pr_author_df,
    on="pullRequestId",
    how="left",
    validate="one_to_one",
)


# Standardize lifecycle timestamps to UTC.

pr_timestamp_columns = [
    "createdAt",
    "updatedAt",
    "closedAt",
    "mergedAt",
]

for column in pr_timestamp_columns:

    pr_lifecycle_df[column] = pd.to_datetime(
        pr_lifecycle_df[column],
        utc=True,
        errors="raise",
    )


# Standardize review attributes and timestamps.

review_lifecycle_df = (
    reviews_preview_df[
        [
            "pullRequestId",
            "review.id",
            "review.user.id",
            "review.user.type",
            "review.state",
            "review.submitted_at",
        ]
    ]
    .copy()
    .rename(
        columns={
            "review.id": "reviewId",
            "review.user.id": "reviewerId",
            "review.user.type": "reviewerType",
            "review.state": "reviewState",
            "review.submitted_at": "submittedAt",
        }
    )
)

review_lifecycle_df["submittedAt"] = pd.to_datetime(
    review_lifecycle_df["submittedAt"],
    utc=True,
    errors="raise",
)


# ------------------------------------------------------------
# 06. Validate lifecycle timestamp consistency
# ------------------------------------------------------------

# Updated timestamps must not precede creation timestamps.

invalid_update_timestamps = (
    pr_lifecycle_df["createdAt"].isna()
    | pr_lifecycle_df["updatedAt"].isna()
    | (
        pr_lifecycle_df["updatedAt"]
        < pr_lifecycle_df["createdAt"]
    )
)

record_validation(
    rule_id="DQ-009",
    rule_name="PR creation and update timestamp consistency",
    records_checked=len(pr_lifecycle_df),
    failed_records=int(invalid_update_timestamps.sum()),
)


# Merged PRs must have a valid merge timestamp.
# Unmerged PRs should not have a populated merge timestamp.

invalid_merge_status = (
    pr_lifecycle_df["isMerged"].isna()
    | pr_lifecycle_df["isMerged"]
        .ne(pr_lifecycle_df["mergedAt"].notna())
        .fillna(True)
)

record_validation(
    rule_id="DQ-010",
    rule_name="Merge status and merge timestamp consistency",
    records_checked=len(pr_lifecycle_df),
    failed_records=int(invalid_merge_status.sum()),
)


# Merge timestamps must not precede PR creation.

merged_records_df = pr_lifecycle_df.loc[
    pr_lifecycle_df["isMerged"].eq(True)
]

invalid_merge_timestamps = (
    merged_records_df["mergedAt"].isna()
    | merged_records_df["createdAt"].isna()
    | (
        merged_records_df["mergedAt"]
        < merged_records_df["createdAt"]
    )
)

record_validation(
    rule_id="DQ-011",
    rule_name="PR creation and merge timestamp consistency",
    records_checked=len(merged_records_df),
    failed_records=int(invalid_merge_timestamps.sum()),
)


# Closed PRs must contain a valid closure timestamp.

closed_records_df = pr_lifecycle_df.loc[
    pr_lifecycle_df["pullRequestState"].eq("closed")
]

invalid_closure_timestamps = (
    closed_records_df["closedAt"].isna()
    | closed_records_df["createdAt"].isna()
    | (
        closed_records_df["closedAt"]
        < closed_records_df["createdAt"]
    )
)

record_validation(
    rule_id="DQ-012",
    rule_name="Closed PR timestamp consistency",
    records_checked=len(closed_records_df),
    failed_records=int(invalid_closure_timestamps.sum()),
)


# ------------------------------------------------------------
# 07. Validate review extraction completeness
# ------------------------------------------------------------

review_coverage_df = (
    pd.DataFrame(
        {"pullRequestNumber": sample_pr_numbers}
    )
    .merge(
        review_extraction_summary_df[
            [
                "pullRequestNumber",
                "isExtractionComplete",
            ]
        ],
        on="pullRequestNumber",
        how="left",
        validate="one_to_one",
    )
)

incomplete_review_extractions = (
    ~review_coverage_df["isExtractionComplete"]
    .fillna(False)
    .astype(bool)
)

record_validation(
    rule_id="DQ-013",
    rule_name="Sample review pagination completeness",
    records_checked=len(review_coverage_df),
    failed_records=int(
        incomplete_review_extractions.sum()
    ),
)


# ------------------------------------------------------------
# 08. Enrich reviews with parent PR lifecycle context
# ------------------------------------------------------------

review_lifecycle_df = review_lifecycle_df.merge(
    pr_lifecycle_df[
        [
            "pullRequestId",
            "authorId",
            "createdAt",
            "mergedAt",
        ]
    ],
    on="pullRequestId",
    how="left",
    validate="many_to_one",
)

submitted_review_mask = (
    review_lifecycle_df["submittedAt"].notna()
)

# A submitted review should not precede PR creation.

invalid_review_timestamps = (
    submitted_review_mask
    & (
        review_lifecycle_df["createdAt"].isna()
        | (
            review_lifecycle_df["submittedAt"]
            < review_lifecycle_df["createdAt"]
        )
    )
)

record_validation(
    rule_id="DQ-014",
    rule_name="Review and PR creation timestamp consistency",
    records_checked=int(submitted_review_mask.sum()),
    failed_records=int(invalid_review_timestamps.sum()),
)


# ------------------------------------------------------------
# 09. Validate non-negative PR activity statistics
# ------------------------------------------------------------

activity_columns = [
    "additionsCount",
    "deletionsCount",
    "changedFilesCount",
    "commitsCount",
]

invalid_activity_values = (
    pr_lifecycle_df[activity_columns]
    .lt(0)
    .any(axis=1)
)

record_validation(
    rule_id="DQ-015",
    rule_name="Non-negative PR activity statistics",
    records_checked=len(pr_lifecycle_df),
    failed_records=int(invalid_activity_values.sum()),
)


# ------------------------------------------------------------
# 10. Generate consolidated validation results
# ------------------------------------------------------------

source_reconciliation_results_df = pd.DataFrame(
    validation_results
)

print("\nEIP | Source Reconciliation Results")
print("=" * 65)

display(source_reconciliation_results_df)

validation_status_summary_df = (
    source_reconciliation_results_df["validationStatus"]
    .value_counts()
    .rename_axis("validationStatus")
    .reset_index(name="ruleCount")
)

print("\nValidation Status Summary:")

display(validation_status_summary_df)


# Stop analytical processing if critical validation fails.

critical_validation_failures = (
    source_reconciliation_results_df["validationStatus"]
    .eq("FAIL")
    .sum()
)

if critical_validation_failures > 0:
    raise RuntimeError(
        "Critical reconciliation failures detected."
    )


# ------------------------------------------------------------
# 11. Define qualifying external reviews
# ------------------------------------------------------------

# Preliminary analytical definition:
#
# - Review must have been submitted.
# - Reviewer and author identifiers must be available.
# - Reviewer must not be the PR author.
# - Reviewer account type must be User.
# - Review must occur after or at PR creation.
# - For merged PRs, review must occur before or at merge.
#
# Dismissed reviews remain historical review events.
# They are not counted as active approvals.
#
# These business rules will later reside in the Control layer.

qualifying_review_states = {
    "APPROVED",
    "CHANGES_REQUESTED",
    "COMMENTED",
    "DISMISSED",
}

review_lifecycle_df["isSelfReview"] = (
    review_lifecycle_df["reviewerId"]
    .eq(review_lifecycle_df["authorId"])
    .fillna(False)
)

review_lifecycle_df["isQualifyingReview"] = (
    review_lifecycle_df["submittedAt"].notna()
    & review_lifecycle_df["reviewerId"].notna()
    & review_lifecycle_df["authorId"].notna()
    & ~review_lifecycle_df["isSelfReview"]
    & review_lifecycle_df["reviewerType"].eq("User")
    & review_lifecycle_df["reviewState"].isin(
        qualifying_review_states
    )
    & (
        review_lifecycle_df["submittedAt"]
        >= review_lifecycle_df["createdAt"]
    )
    & (
        review_lifecycle_df["mergedAt"].isna()
        | (
            review_lifecycle_df["submittedAt"]
            <= review_lifecycle_df["mergedAt"]
        )
    )
).fillna(False)


# ------------------------------------------------------------
# 12. Aggregate qualifying reviews to pull request grain
# ------------------------------------------------------------

# Aggregate before joining to avoid inflating PR counts
# and change-size metrics through a one-to-many join.

qualifying_reviews_df = review_lifecycle_df.loc[
    review_lifecycle_df["isQualifyingReview"]
]

review_aggregation_df = (
    qualifying_reviews_df
    .groupby("pullRequestId", as_index=False)
    .agg(
        firstReviewAt=("submittedAt", "min"),
        lastQualifyingReviewAt=("submittedAt", "max"),
        externalReviewCount=("reviewId", "nunique"),
        uniqueExternalReviewerCount=(
            "reviewerId",
            "nunique",
        ),
    )
)


# ------------------------------------------------------------
# 13. Build PR lifecycle analytical prototype
# ------------------------------------------------------------

pr_lifecycle_prototype_df = (
    pr_lifecycle_df
    .merge(
        review_aggregation_df,
        on="pullRequestId",
        how="left",
        validate="one_to_one",
    )
    .merge(
        review_coverage_df,
        on="pullRequestNumber",
        how="left",
        validate="one_to_one",
    )
)

pr_lifecycle_prototype_df["repositoryId"] = (
    repository_raw["id"]
)

review_count_columns = [
    "externalReviewCount",
    "uniqueExternalReviewerCount",
]

# Zero is valid only when the sampled review extraction
# is complete. Unknown coverage must not become zero.

complete_review_mask = (
    pr_lifecycle_prototype_df["isExtractionComplete"]
    .fillna(False)
    .astype(bool)
)

for column in review_count_columns:

    pr_lifecycle_prototype_df[column] = (
        pr_lifecycle_prototype_df[column]
        .fillna(0)
        .astype("Int64")
        .where(complete_review_mask, pd.NA)
    )

for column in [
    "firstReviewAt",
    "lastQualifyingReviewAt",
]:

    pr_lifecycle_prototype_df[column] = (
        pr_lifecycle_prototype_df[column]
        .where(complete_review_mask, pd.NaT)
    )


# ------------------------------------------------------------
# 14. Calculate deterministic lifecycle metrics
# ------------------------------------------------------------

SECONDS_PER_HOUR = 3600

# Total number of lines added and removed.

pr_lifecycle_prototype_df["linesChanged"] = (
    pr_lifecycle_prototype_df["additionsCount"]
    + pr_lifecycle_prototype_df["deletionsCount"]
)


# Elapsed time from creation to successful merge.

pr_lifecycle_prototype_df["mergeHours"] = (
    (
        pr_lifecycle_prototype_df["mergedAt"]
        - pr_lifecycle_prototype_df["createdAt"]
    )
    .dt.total_seconds()
    / SECONDS_PER_HOUR
)


# Elapsed time from creation to first qualifying review.

pr_lifecycle_prototype_df["preReviewHours"] = (
    (
        pr_lifecycle_prototype_df["firstReviewAt"]
        - pr_lifecycle_prototype_df["createdAt"]
    )
    .dt.total_seconds()
    / SECONDS_PER_HOUR
)


# Elapsed time from first qualifying review to merge.
# Remains null for unmerged or unreviewed PRs.

pr_lifecycle_prototype_df["postReviewHours"] = (
    (
        pr_lifecycle_prototype_df["mergedAt"]
        - pr_lifecycle_prototype_df["firstReviewAt"]
    )
    .dt.total_seconds()
    / SECONDS_PER_HOUR
)


# Calculate the share of total merge duration occurring
# before the first qualifying review.
# Avoid division by zero.

valid_merge_duration = (
    pr_lifecycle_prototype_df["mergeHours"]
    .where(
        pr_lifecycle_prototype_df["mergeHours"] > 0
    )
)

pr_lifecycle_prototype_df["preReviewTimeSharePct"] = (
    pr_lifecycle_prototype_df["preReviewHours"]
    .div(valid_merge_duration)
    .mul(100)
)


# ------------------------------------------------------------
# 15. Display PR lifecycle analytical dataset
# ------------------------------------------------------------

lifecycle_output_columns = [
    "repositoryId",
    "pullRequestId",
    "pullRequestNumber",
    "pullRequestState",
    "isDraft",
    "isMerged",
    "linesChanged",
    "externalReviewCount",
    "uniqueExternalReviewerCount",
    "firstReviewAt",
    "mergeHours",
    "preReviewHours",
    "postReviewHours",
    "preReviewTimeSharePct",
]

print("\nEIP | PR Lifecycle Analytical Prototype")
print("=" * 65)

# Round analytical measures for presentation only.
# Preserve original timestamp and identifier precision.

lifecycle_display_df = (
    pr_lifecycle_prototype_df[
        lifecycle_output_columns
    ]
    .copy()
)

lifecycle_metric_columns = [
    "mergeHours",
    "preReviewHours",
    "postReviewHours",
    "preReviewTimeSharePct",
]

lifecycle_display_df[lifecycle_metric_columns] = (
    lifecycle_display_df[lifecycle_metric_columns]
    .round(4)
)

display(lifecycle_display_df)


# ------------------------------------------------------------
# 16. Validate lifecycle metric reconciliation
# ------------------------------------------------------------

# For reviewed and merged PRs:
#
# mergeHours = preReviewHours + postReviewHours
#
# A small tolerance accounts for floating-point arithmetic.

reviewed_merged_prs_df = (
    pr_lifecycle_prototype_df
    .dropna(
        subset=[
            "mergeHours",
            "preReviewHours",
            "postReviewHours",
        ]
    )
    .copy()
)

reviewed_merged_prs_df["lifecycleDifferenceHours"] = (
    reviewed_merged_prs_df["mergeHours"]
    - reviewed_merged_prs_df["preReviewHours"]
    - reviewed_merged_prs_df["postReviewHours"]
).abs()

LIFECYCLE_TOLERANCE = 1e-9

record_validation(
    rule_id="DQ-016",
    rule_name="Lifecycle duration reconciliation",
    records_checked=len(reviewed_merged_prs_df),
    failed_records=int(
        reviewed_merged_prs_df["lifecycleDifferenceHours"]
        .gt(LIFECYCLE_TOLERANCE)
        .sum()
    ),
)

source_reconciliation_results_df = pd.DataFrame(
    validation_results
)


# ------------------------------------------------------------
# 17. Display final validation summary
# ------------------------------------------------------------

print("\nEIP | Final Reconciliation Summary")
print("=" * 65)

display(source_reconciliation_results_df)

print("Total Validation Rules:", len(validation_results))

print(
    "Passed:",
    int(
        source_reconciliation_results_df["validationStatus"]
        .eq("PASS")
        .sum()
    ),
)

print(
    "Warnings:",
    int(
        source_reconciliation_results_df["validationStatus"]
        .eq("WARNING")
        .sum()
    ),
)

print(
    "Failed:",
    int(
        source_reconciliation_results_df["validationStatus"]
        .eq("FAIL")
        .sum()
    ),
)

print(
    "Not Assessed:",
    int(
        source_reconciliation_results_df["validationStatus"]
        .eq("NOT_ASSESSED")
        .sum()
    ),
)

print(
    "Lifecycle Records Generated:",
    len(pr_lifecycle_prototype_df),
)

print(
    "Qualifying External Reviews:",
    len(qualifying_reviews_df),
)

print("Historical Backfill Completed: False")
print("Database Persistence Completed: False")

EIP | PR Attribute Reconciliation


,sourceAttribute,recordsCompared,matchingRecords,mismatchedRecords
0,state,3,3,0
1,draft,3,3,0
2,created_at,3,3,0
3,updated_at,3,3,0
4,closed_at,3,3,0
5,merged_at,3,3,0



EIP | Source Reconciliation Results


,ruleId,ruleName,severity,recordsChecked,failedRecords,validationStatus
0,DQ-001,Repository identifier,ERROR,1,0,PASS
1,DQ-002,Pull request identifier,ERROR,10,0,PASS
2,DQ-003,Pull request detail identifier,ERROR,3,0,PASS
3,DQ-004,Review identifier,ERROR,22,0,PASS
4,DQ-005,PR details to pull requests,ERROR,3,0,PASS
5,DQ-006,Reviews to parent pull requests,ERROR,22,0,PASS
6,DQ-007,Reviews to parent repository,ERROR,22,0,PASS
7,DQ-008,PR list and detail attribute consistency,WARNING,18,0,PASS
8,DQ-009,PR creation and update timestamp consistency,ERROR,3,0,PASS
9,DQ-010,Merge status and merge timestamp consistency,ERROR,3,0,PASS



Validation Status Summary:


,validationStatus,ruleCount
0,PASS,15



EIP | PR Lifecycle Analytical Prototype


,repositoryId,pullRequestId,pullRequestNumber,pullRequestState,isDraft,isMerged,linesChanged,externalReviewCount,uniqueExternalReviewerCount,firstReviewAt,mergeHours,preReviewHours,postReviewHours,preReviewTimeSharePct
0,33884891,5505694791,77624,closed,False,True,89,3,3,2026-11-26 07:48:53+00:00,2.3469,0.5528,1.7942,23.5531
1,33884891,4049905904,69849,open,False,False,442,14,2,2026-07-14 09:55:55+00:00,NaN,4.4475,NaN,NaN
2,33884891,4148360837,70570,open,True,False,5654,2,2,2026-11-23 23:26:21+00:00,NaN,2850.5456,NaN,NaN



EIP | Final Reconciliation Summary


,ruleId,ruleName,severity,recordsChecked,failedRecords,validationStatus
0,DQ-001,Repository identifier,ERROR,1,0,PASS
1,DQ-002,Pull request identifier,ERROR,10,0,PASS
2,DQ-003,Pull request detail identifier,ERROR,3,0,PASS
3,DQ-004,Review identifier,ERROR,22,0,PASS
4,DQ-005,PR details to pull requests,ERROR,3,0,PASS
5,DQ-006,Reviews to parent pull requests,ERROR,22,0,PASS
6,DQ-007,Reviews to parent repository,ERROR,22,0,PASS
7,DQ-008,PR list and detail attribute consistency,WARNING,18,0,PASS
8,DQ-009,PR creation and update timestamp consistency,ERROR,3,0,PASS
9,DQ-010,Merge status and merge timestamp consistency,ERROR,3,0,PASS


Total Validation Rules: 16
Passed: 16
Warnings: 0
Failed: 0
Not Assessed: 0
Lifecycle Records Generated: 3
Qualifying External Reviews: 19
Historical Backfill Completed: False
Database Persistence Completed: False


# 09. Source Exploration Findings and Conclusions

## 9.1 Executive Summary

Successfully completed the initial GitHub REST API source
exploration for the Engineering Intelligence Platform (EIP).

All four planned source entities were retrieved, profiled
and reconciled using authenticated, read-only API access.

The exploration confirmed the availability of the source
attributes required for pull request lifecycle analysis,
engineering collaboration metrics and evidence-grounded
AI intelligence.

## 9.2 Validated Source Inventory

| Source ID | Source Entity | Records | Discovered Attributes | Selected Attributes |
|---|---|---:|---:|---:|
| SRC-001 | Repositories | 1 | 128 | 17 |
| SRC-002 | Pull Requests | 10 | 361 | 15 |
| SRC-003 | Pull Request Details | 3 | 392 | 17 |
| SRC-004 | Pull Request Reviews | 22 | 35 | 12 |
| Total | All Sources | 36 | 916 | 61 |

All selected attributes were observed in the extracted sample.

The discovered attribute count represents the sum of flattened
attributes across source entities, including ingestion context.

## 9.3 Source-to-Target Architecture

### Bronze Layer

| Source ID | Proposed Bronze Table |
|---|---|
| SRC-001 | eip_tbl_bronze_githubRepositories_daily |
| SRC-002 | eip_tbl_bronze_githubPullRequests_daily |
| SRC-003 | eip_tbl_bronze_githubPullRequestDetails_daily |
| SRC-004 | eip_tbl_bronze_githubReviews_daily |

The Bronze layer will preserve original source JSON records,
source identifiers, ingestion timestamps, execution identifiers
and payload hashes.

### Silver Layer

| Proposed Silver Table | Grain |
|---|---|
| eip_tbl_silver_repositories_daily | One current record per repository |
| eip_tbl_silver_pullRequests_daily | One current record per pull request |
| eip_tbl_silver_reviews_daily | One current record per review |

Pull request list and detail records will be consolidated
into a single enriched Silver pull request entity.

### Gold Layer

| Proposed Gold Table | Grain |
|---|---|
| eip_tbl_gold_prLifecycle_wide | One analytical record per pull request |
| eip_tbl_gold_periodFacts_long | One record per repository, reporting period and metric |

### Control Layer

Static source definitions, metric definitions, business rules
and data quality configurations will be maintained through
version-controlled SQL views.

Operational execution history, ingestion watermarks and
validation results will be persisted in control tables.

The proposed object names describe the target architecture.
Physical database objects and refresh jobs are not yet implemented.

## 9.4 Data Quality Results

| Validation Category | Result |
|---|---:|
| Selected Attributes | 61 |
| Attributes Available | 61 |
| Missing Selected Attributes | 0 |
| Attributes Without Observed Nulls | 54 |
| Attributes With Potentially Valid Nulls | 7 |
| Critical Attribute Quality Issues | 0 |
| Reconciliation Rules Executed | 16 |
| Reconciliation Rules Passed | 16 |
| Reconciliation Rules Failed | 0 |
| Reconciliation Rule Warnings | 0 |
| Reconciliation Pass Rate | 100% |

Validated quality dimensions include:

- Source identifier completeness and uniqueness
- Parent-child referential integrity
- Cross-endpoint attribute consistency
- Lifecycle timestamp consistency
- Review extraction completeness
- Non-negative activity statistics
- Lifecycle duration reconciliation

These results apply to the extracted exploratory sample.

## 9.5 Key Source Observations

### Source Identifier Precision

GitHub pull request and review identifiers exceed the
signed 32-bit integer range.

Source identifiers must be preserved using 64-bit integers.

Nullable identifiers must not be converted through
floating-point representations during transformations.

### Source Nullability

Missing closure and merge timestamps are expected for
open or unmerged pull requests.

Missing merge account information may occur for
unmerged pull requests.

Seven of the 22 review records contained missing
commit identifiers.

Review commit identifiers must therefore support null values.

### Historical Coverage

Several sampled pull requests were created outside the
proposed 90-day reporting window but were recently updated.

Historical ingestion must consider relevant lifecycle
activity rather than filtering exclusively by creation date.

### Collaboration Semantics

General comments, code review comments and submitted
reviews represent different source entities or measures.

They must not be treated as interchangeable metrics.

### Review Qualification

The exploratory analytical definition excludes:

- Self-reviews
- Accounts explicitly classified as bots
- Pending or unsubmitted reviews
- Reviews preceding pull request creation
- Reviews occurring after merge

Dismissed reviews are retained as historical collaboration
events but do not represent active approvals.

GitHub account type is an imperfect proxy for human activity.

## 9.6 Gold Analytical Prototype

| Metric | PR 77624 | PR 69849 | PR 70570 |
|---|---:|---:|---:|
| Lines Changed | 89 | 442 | 5654 |
| Qualifying External Reviews | 3 | 14 | 2 |
| Unique External Reviewers | 3 | 2 | 2 |
| Merge Hours | 2.3469 | NULL | NULL |
| Pre-Review Hours | 0.5528 | 4.4475 | 2850.5456 |
| Post-Review Hours | 1.7942 | NULL | NULL |
| Pre-Review Time Share (%) | 23.5531 | NULL | NULL |

The analytical prototype was generated through deterministic
transformations of the extracted source datasets.

Review records were aggregated to pull request grain before
joining to prevent duplication of pull request measures.

## 9.7 Preliminary Metric Feasibility

| Metric | Feasibility |
|---|---|
| Pull Requests Opened | Source attributes available |
| Pull Requests Merged | Source attributes available |
| Median Merge Duration | Source attributes available |
| Time to First External Review | Demonstrated |
| Post-Review Merge Duration | Demonstrated |
| Pre-Review Time Share | Demonstrated |
| Pull Request Change Size | Demonstrated |
| External Review Count | Demonstrated |
| Unique External Reviewers | Demonstrated |
| Review Coverage | Source attributes available |
| Reviewer Concentration | Source attributes available |
| Historical Benchmarks | Pending historical extraction |
| Anomaly Detection | Pending historical extraction |

Feasibility does not imply that production metric definitions,
historical coverage or statistical reliability are finalized.

## 9.8 Exploration Limitations

- Only one public GitHub repository was explored.
- Pull request extraction was limited to a bounded sample.
- Details and reviews were retrieved for three selected PRs.
- The sample was deliberately selected and is not representative.
- Historical reporting coverage has not been established.
- Current draft status does not reconstruct historical readiness.
- Elapsed lifecycle time does not measure engineering effort.
- Source responses were retrieved at different points in time.
- Physical database persistence has not been implemented.
- Production retry and incremental ingestion logic are pending.

## 9.9 Engineering and Governance Requirements

All production database objects must document:

- Object purpose and business description
- Object grain
- Primary and foreign keys
- Column names and data types
- Column-level descriptions
- Nullability and constraints
- Source-to-target lineage
- Transformation logic
- Refresh and retention strategies

All production code must include appropriate module headers,
function documentation, type hints and meaningful comments.

## 9.10 Next Steps

1. Profile historical pull request activity and volume.
2. Estimate API request consumption.
3. Finalize the historical ingestion window.
4. Validate period-level metric feasibility.
5. Finalize source-to-target mappings.
6. Develop static source and metric registry control views.
7. Complete the centralized data dictionary.
8. Implement the production GitHub API client.
9. Implement ingestion client tests.
10. Implement persistent Bronze, Silver and Gold layers.

## 9.11 Completion Status

Initial Source Exploration: COMPLETE

Historical Data Profiling: PENDING

Production API Client: PENDING

Database Persistence: PENDING
